In [ ]:
import os
import sys
from google.colab import drive

# Monta Drive
drive.mount('/content/drive')

# Configura Repo
REPO_NAME = "counter_sspa"
REPO_URL = f"https://github.com/satia2/{REPO_NAME}.git"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    %cd {REPO_NAME}
    !git pull origin main
    %cd ..

# Aggiungi al path
path_repo = os.path.abspath(REPO_NAME)
if path_repo not in sys.path:
    sys.path.append(path_repo)

# Installazione librerie (se necessario)
!pip install -r {REPO_NAME}/requirements.txt
!pip install cellpose torch torchvision

In [ ]:
import torch
from cellpose import models
import importlib

# Carichiamo il modello globalmente
usa_gpu = torch.cuda.is_available()
model_ia = models.CellposeModel(model_type='cyto', gpu=usa_gpu)
print(f"✅ Modello IA caricato! GPU attiva: {usa_gpu}")

In [ ]:
import importlib
import src.counter
importlib.reload(src.counter)
from src.counter import conta_colonie
print("✅ Funzioni aggiornate con le ultime modifiche.")

In [ ]:
import cv2
import matplotlib.pyplot as plt

input_path = '/content/drive/MyDrive/SaggioClonogenico/input'
output_path = '/content/drive/MyDrive/SaggioClonogenico/output'

if not os.path.exists(output_path): os.makedirs(output_path)

immagini = [f for f in os.listdir(input_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

for nome_file in immagini:
    img = cv2.imread(os.path.join(input_path, nome_file))
    if img is None: continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Passiamo il modello già caricato (model_ia) alla funzione
    n, maschera = conta_colonie(img_rgb, model_ia, diametro=21)
    
    plt.figure(figsize=(8, 8))
    plt.imshow(img_rgb)
    plt.imshow(maschera, alpha=0.3, cmap='prism')
    plt.title(f"{nome_file}: {n} colonie")
    plt.axis('off')
    plt.savefig(os.path.join(output_path, f"risultato_{nome_file}"), bbox_inches='tight')
    plt.show()
    print(f"✅ {nome_file}: {n} colonie.")